In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')



In [8]:
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
features = pd.read_csv("../data/features.csv")

In [4]:
def apply_rules(row):
    rules_triggered = []
    
    # Rule 1: High event velocity
    if row["events_per_day"] > 150:
        rules_triggered.append("high_velocity")
    
    # Rule 2: IP shared by many accounts
    if row["users_sharing_ip"] > 10:
        rules_triggered.append("shared_ip")
    
    # Rule 3: New account with no history
    if row["account_age_days"] < 10 and row["subscriber_count"] < 20:
        rules_triggered.append("new_account_no_history")
    
    # Rule 4: Only likes and views, no comments or shares
    if row["ratio_comment"] == 0 and row["ratio_share"] == 0:
        rules_triggered.append("no_comments_or_shares")
    
    return pd.Series({
        "rules_triggered" : rules_triggered,
        "rule_score"      : len(rules_triggered),
        "is_flagged"      : len(rules_triggered) > 0,
    })



In [5]:
# Apply to every user
rule_results = features.apply(apply_rules, axis=1)
features = pd.concat([features, rule_results], axis=1)

print(features[["user_id","is_bot","rule_score","is_flagged","rules_triggered"]].head(5))

     user_id  is_bot  rule_score  is_flagged rules_triggered
0  USR_00000       0           0       False              []
1  USR_00001       0           0       False              []
2  USR_00002       0           0       False              []
3  USR_00003       0           0       False              []
4  USR_00004       0           0       False              []


In [6]:
features.to_csv("../data/features_with_rules.csv", index=False)

In [6]:
print("Bot accounts:")
print(features[features["is_bot"]==1][["user_id","rule_score","is_flagged","rules_triggered"]].head(5))

Bot accounts:
       user_id  rule_score  is_flagged  \
800  BOT_00000           4        True   
801  BOT_00001           3        True   
802  BOT_00002           4        True   
803  BOT_00003           3        True   
804  BOT_00004           3        True   

                                       rules_triggered  
800  [high_velocity, shared_ip, new_account_no_hist...  
801  [high_velocity, new_account_no_history, no_com...  
802  [high_velocity, shared_ip, new_account_no_hist...  
803  [high_velocity, new_account_no_history, no_com...  
804  [high_velocity, new_account_no_history, no_com...  


## Evaluate the rule engine


In [9]:
print("=== Rule Engine Performance ===\n")
print(classification_report(features["is_bot"], features["is_flagged"]))

print("Confusion Matrix:")
print(confusion_matrix(features["is_bot"], features["is_flagged"]))


=== Rule Engine Performance ===

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       800
           1       1.00      1.00      1.00       200

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Confusion Matrix:
[[800   0]
 [  0 200]]


In [10]:
from collections import Counter

all_rules = []
for rules in features["rules_triggered"]:
    all_rules.extend(rules)

rule_counts = Counter(all_rules)
print("Rules fired count:")
for rule, count in rule_counts.most_common():
    print(f"  {rule}: {count} times")

Rules fired count:
  high_velocity: 200 times
  new_account_no_history: 200 times
  no_comments_or_shares: 200 times
  shared_ip: 122 times
